# AorticTurb — Pass 3 canonical Colab runner

This notebook is a thin orchestration layer over **AorticTurb**. Scientific calculations remain in the tested package.

Pass 3 executes only the agreed sequence: verified staging/startup gate → fresh phase-conditioned decomposition → Gate A with the unchanged legacy primary support rule → the pre-specified support-policy sensitivity analysis.

It does not call the older `run_all` workflow, does not promote outputs to canonical `05_results`, and does not automatically enter downstream KO/Taylor/Lumley/diagnostics/classification stages.

Locked Pass-3 runner/package commit: `f48d6564fa29231a432543f721ef0448f719c3b9`

Persistent outputs are written under `<Google Drive project root>/results/`; canonical raw Drive data are never modified.


In [ ]:
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

AORTICTURB_REPO = "https://github.com/khalid-saqr/AorticTurb.git"
AORTICTURB_COMMIT = "f48d6564fa29231a432543f721ef0448f719c3b9"
EXPECTED_AORTICTURB_VERSION = "0.2.0"
SAFE_CWD = Path("/content")
WORKTREE = SAFE_CWD / "AorticTurb"

if sys.version_info[:2] not in {(3, 10), (3, 11), (3, 12), (3, 13)}:
    raise RuntimeError(f"UNSUPPORTED_PYTHON: {sys.version.split()[0]}")
if not SAFE_CWD.is_dir():
    raise RuntimeError(f"SAFE_WORKING_DIRECTORY_MISSING: {SAFE_CWD}")
# A previous notebook run may have left the process cwd inside WORKTREE.
# Move out before deleting that tree so reruns never inherit a deleted cwd.
os.chdir(SAFE_CWD)
if not (AORTICTURB_REPO.startswith("https://github.com/") and AORTICTURB_REPO.endswith(".git")) or any(ch in AORTICTURB_REPO for ch in "[]()"):
    raise RuntimeError(f"INVALID_REPOSITORY_URL: expected raw Git URL, got {AORTICTURB_REPO!r}")

def _remove_worktree():
    os.chdir(SAFE_CWD)
    if WORKTREE.exists():
        if not str(WORKTREE).startswith("/content/"):
            raise RuntimeError(f"REFUSING_TO_REMOVE_NON_SCRATCH_PATH: {WORKTREE}")
        shutil.rmtree(WORKTREE)

def _git(args, *, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)

clone_errors = []
for attempt in range(1, 4):
    _remove_worktree()
    clone = _git(["clone", "--no-tags", "--filter=blob:none", AORTICTURB_REPO, str(WORKTREE)], cwd=SAFE_CWD)
    if clone.returncode == 0:
        break
    clone_errors.append(f"attempt {attempt}: {clone.stderr.strip() or clone.stdout.strip() or 'git clone failed without output'}")
    if attempt < 3:
        time.sleep(2 * attempt)
else:
    raise RuntimeError("GIT_CLONE_FAILED for repository " + repr(AORTICTURB_REPO) + "\n" + "\n".join(clone_errors))

checkout = _git(["checkout", "--detach", AORTICTURB_COMMIT], cwd=WORKTREE)
if checkout.returncode != 0:
    raise RuntimeError(f"GIT_CHECKOUT_FAILED for commit {AORTICTURB_COMMIT}: {checkout.stderr.strip() or checkout.stdout.strip()}")
observed = _git(["rev-parse", "HEAD"], cwd=WORKTREE)
if observed.returncode != 0:
    raise RuntimeError(f"GIT_REV_PARSE_FAILED: {observed.stderr.strip() or observed.stdout.strip()}")
observed_commit = observed.stdout.strip()
if observed_commit != AORTICTURB_COMMIT:
    raise RuntimeError(f"GIT_COMMIT_MISMATCH: expected {AORTICTURB_COMMIT}, got {observed_commit}")

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "--force-reinstall", "--no-deps", str(WORKTREE)], check=True)
os.chdir(WORKTREE)

# Colab can retain an older aorticturb import across notebook reruns. Purge it
# explicitly, then put this detached checkout first on sys.path so the code
# actually executed is the code identified by AORTICTURB_COMMIT.
for module_name in list(sys.modules):
    if module_name == "aorticturb" or module_name.startswith("aorticturb."):
        del sys.modules[module_name]
source_root = (WORKTREE / "src").resolve()
sys.path = [p for p in sys.path if Path(p or ".").resolve() != source_root]
sys.path.insert(0, str(source_root))
importlib.invalidate_caches()

import aorticturb
loaded_package = Path(aorticturb.__file__).resolve()
if source_root not in loaded_package.parents:
    raise RuntimeError(f"AORTICTURB_IMPORT_PROVENANCE_FAILED: loaded {loaded_package}, expected source under {source_root}")
if aorticturb.__version__ != EXPECTED_AORTICTURB_VERSION:
    raise RuntimeError(f"INSTALL_FAILED: expected {EXPECTED_AORTICTURB_VERSION}, got {aorticturb.__version__}")

runner_path = WORKTREE / "notebooks" / "pass3_runner.py"
spec = importlib.util.spec_from_file_location("aorticturb_pass3_runner", runner_path)
runner = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(runner)

print("AorticTurb", aorticturb.__version__)
print("Loaded package", loaded_package)
print("Pass-3 runner/package commit", observed_commit)


In [ ]:
from IPython.display import display
import pandas as pd

# EDIT ONLY THIS PATH IF THE PROJECT IS MOUNTED ELSEWHERE.
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study"

pass3 = runner.run_pass3(
    expected_commit=AORTICTURB_COMMIT,
    drive_project_root=DRIVE_PROJECT_ROOT,
)

display(pd.DataFrame(pass3["support_table"]))
display(pd.DataFrame(pass3["sensitivity_table"]))
print("Primary Gate A:", pass3["primary_gate_a"])
print("Pass-3 record:", pass3["record_dir"])
print("Canonical 05_results promoted:", pass3["canonical_05_results_promoted"])
